# Face Recognition Project
## Using Pre-trained CNN Models

This notebook demonstrates two approaches to face recognition:
1. **Easy Approach**: Using `face_recognition` library (dlib's ResNet)
2. **Deep Learning Approach**: Using VGG-Face pre-trained model

### Dataset Recommendations:
- **Labeled Faces in the Wild (LFW)**: Download from http://vis-www.cs.umass.edu/lfw/
- **Custom Dataset**: Create folders with person names and add their images

### Installation Requirements:
```bash
pip install face_recognition opencv-python pillow numpy
pip install tensorflow keras-vggface keras-applications
```

---
## Using VGG-Face Pre-trained Model
This approach uses the VGG-Face model with TensorFlow/Keras

In [ ]:
# Import additional libraries for VGG-Face
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from keras_vggface.vggface import VGGFace
from keras_vggface.utils import preprocess_input
from tensorflow.keras.preprocessing import image
from scipy.spatial.distance import cosine

print(f"TensorFlow version: {tf.__version__}")
print("VGG-Face libraries imported successfully!")

### Load VGG-Face Pre-trained Model

In [ ]:
# Load VGG-Face model (this will download the model on first run)
# Using ResNet50 architecture
vgg_model = VGGFace(model='resnet50', include_top=False, input_shape=(224, 224, 3), pooling='avg')

print("VGG-Face model loaded successfully!")
print(f"Model input shape: {vgg_model.input_shape}")
print(f"Model output shape: {vgg_model.output_shape}")

### Extract Face Embeddings with VGG-Face

In [ ]:
def extract_face_vgg(image_path, required_size=(224, 224)):
    """
    Extract and preprocess face from image for VGG-Face
    """
    # Load image
    img = image.load_img(image_path, target_size=required_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array, version=2)  # version=2 for ResNet50
    return img_array

def get_face_embedding(model, face_pixels):
    """
    Get face embedding using VGG-Face model
    """
    embedding = model.predict(face_pixels, verbose=0)
    return embedding[0]

def load_faces_vgg(dataset_path, model):
    """
    Load all faces and create embeddings using VGG-Face
    """
    face_embeddings = []
    face_names = []
    
    for person_name in os.listdir(dataset_path):
        person_folder = os.path.join(dataset_path, person_name)
        
        if not os.path.isdir(person_folder):
            continue
        
        print(f"Processing: {person_name}")
        
        for image_name in os.listdir(person_folder):
            if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(person_folder, image_name)
                
                try:
                    face_pixels = extract_face_vgg(image_path)
                    embedding = get_face_embedding(model, face_pixels)
                    face_embeddings.append(embedding)
                    face_names.append(person_name)
                    print(f"  ✓ Processed: {image_name}")
                except Exception as e:
                    print(f"  ✗ Error: {image_name} - {str(e)}")
    
    print(f"\nTotal embeddings created: {len(face_embeddings)}")
    return np.array(face_embeddings), face_names

# Load faces with VGG-Face
if len(known_face_encodings) > 0:  # Only if dataset exists
    vgg_embeddings, vgg_names = load_faces_vgg(DATASET_PATH, vgg_model)
else:
    print("⚠️  Add images to dataset folder first!")

### Face Recognition with VGG-Face

In [ ]:
def recognize_face_vgg(test_image_path, model, known_embeddings, known_names, threshold=0.5):
    """
    Recognize face using VGG-Face embeddings
    
    Args:
        threshold: Cosine distance threshold (lower = more strict)
    """
    # Get embedding for test image
    test_pixels = extract_face_vgg(test_image_path)
    test_embedding = get_face_embedding(model, test_pixels)
    
    # Calculate distances to all known faces
    distances = []
    for known_embedding in known_embeddings:
        distance = cosine(test_embedding, known_embedding)
        distances.append(distance)
    
    # Find best match
    min_distance = min(distances)
    min_index = distances.index(min_distance)
    
    if min_distance <= threshold:
        name = known_names[min_index]
        confidence = (1 - min_distance) * 100
    else:
        name = "Unknown"
        confidence = 0
    
    return name, confidence, min_distance

# Test VGG-Face recognition
if os.path.exists(test_image) and 'vgg_embeddings' in locals():
    name, conf, dist = recognize_face_vgg(test_image, vgg_model, vgg_embeddings, vgg_names)
    
    # Display result
    img = plt.imread(test_image)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"VGG-Face Recognition\nName: {name}\nConfidence: {conf:.1f}%\nDistance: {dist:.3f}")
    plt.show()
    
    print(f"Predicted: {name} (confidence: {conf:.1f}%, distance: {dist:.3f})")
else:
    print("⚠️  Test image not found or embeddings not created!")

---
## Comparison and Analysis

In [ ]:
print("="*60)
print("FACE RECOGNITION COMPARISON")
print("="*60)
print("\nApproach 1: face_recognition (dlib ResNet)")
print("  Pros: Easy to use, good accuracy, handles multiple faces")
print("  Cons: Slower on large datasets")
print("  Best for: Quick prototyping, multiple faces in image")

print("\nApproach 2: VGG-Face (Deep CNN)")
print("  Pros: State-of-art accuracy, customizable, fast inference")
print("  Cons: Requires preprocessing, single face per image")
print("  Best for: Production systems, large-scale deployment")

print("\n" + "="*60)
print("RECOMMENDATION FOR YOUR PROJECT")
print("="*60)
print("✓ Use Approach 1 (face_recognition) for easier implementation")
print("✓ Use Approach 2 (VGG-Face) for better accuracy and control")
print("\nDataset: Start with LFW or create custom dataset with 5-10 images per person")

---
## Download LFW Dataset (Optional)

In [ ]:
# Uncomment to download LFW dataset
# from sklearn.datasets import fetch_lfw_people
# 
# # Download LFW dataset
# lfw_people = fetch_lfw_people(min_faces_per_person=20, resize=0.4)
# 
# print(f"Total dataset size: {lfw_people.images.shape[0]}")
# print(f"Number of classes: {len(lfw_people.target_names)}")
# print(f"Image shape: {lfw_people.images.shape[1:]}")
# print(f"\nPeople: {lfw_people.target_names}")

---
## Save Model/Embeddings (Optional)

In [ ]:
import pickle

# Save encodings for future use
def save_encodings(encodings, names, filename='face_encodings.pkl'):
    """
    Save face encodings and names to file
    """
    data = {'encodings': encodings, 'names': names}
    with open(filename, 'wb') as f:
        pickle.dump(data, f)
    print(f"Encodings saved to {filename}")

def load_encodings(filename='face_encodings.pkl'):
    """
    Load face encodings and names from file
    """
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    print(f"Encodings loaded from {filename}")
    return data['encodings'], data['names']

# Example usage:
# save_encodings(known_face_encodings, known_face_names)
# known_face_encodings, known_face_names = load_encodings()

---
## Summary and Next Steps

### What you've learned:
1. ✓ Using pre-trained CNN models for face recognition
2. ✓ Two different approaches (face_recognition and VGG-Face)
3. ✓ Creating face embeddings/encodings
4. ✓ Testing and evaluating models

### Next steps for your project:
1. **Create your dataset**: Add 5-10 images per person in the dataset folder
2. **Test thoroughly**: Try different people and lighting conditions
3. **Tune parameters**: Adjust tolerance/threshold for better accuracy
4. **Add features**: 
   - Real-time webcam recognition
   - Face detection with anti-spoofing
   - Database integration
   - Attendance system

### Recommended datasets:
- **LFW**: http://vis-www.cs.umass.edu/lfw/ (13K images, 6K people)
- **CelebA**: https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html (200K images)
- **UTKFace**: https://susanqq.github.io/UTKFace/ (20K images with age/gender)
- **Yale Faces**: http://vision.ucsd.edu/content/yale-face-database (Good for small projects)

Good luck with your project! 🚀